In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import time
import os, re, glob
from pyprojroot import here
import resource
import datetime

/home/leon/miniconda3/envs/mouse_brain_cells/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/leon/miniconda3/envs/mouse_brain_cells/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
base_data_folder = "/vault/lfrench/temp_transfer/retina_cellxgene/"

In [3]:
h5ad_files = [
base_data_folder + "LiRetina_A.h5ad",
base_data_folder + "LiRetina_B.h5ad"
]

In [4]:
        adata_a = sc.read_h5ad(base_data_folder + "LiRetina_A.h5ad", backed=True)
        adata_b = sc.read_h5ad(base_data_folder + "LiRetina_B.h5ad", backed=True)
        
        adata_a.obs["study_id"] = adata_a.obs["study_id"].astype(str) + "_A" 
        adata_b.obs["study_id"] = adata_b.obs["study_id"].astype(str) + "_B"


In [5]:
        adata_a = adata_a.to_memory()
        print("Loaded A into memory")
#        adata_b = adata_b[idx, :].to_memory()
        adata_b = adata_b.to_memory()
        print("Done loading files")
        
        merged=ad.concat([adata_a, adata_b], join="inner")
        
        pymn.variableGenes(merged, study_col='study_id')
        merged = merged[:, merged.var.highly_variable]
        
        merged.obs['cell.type'] = merged.obs['cell.type'].to_numpy(dtype="str")
        merged.obs['study_id'] = merged.obs['study_id'].to_numpy(dtype="str")
        merged.obs.index = merged.obs.index.to_numpy(dtype="str")
        #use numpy types instead of pandas
        merged.var.highly_variable = merged.var.highly_variable.to_numpy(dtype="bool")
        merged.var.index = merged.var.index.to_numpy(dtype="str")
        merged.obs_names_make_unique()


Loaded A into memory
Done loading files


/tmp/ipykernel_3100600/3799429671.py:12: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  merged.obs['cell.type'] = merged.obs['cell.type'].to_numpy(dtype="str")


In [6]:
#write out merged?

In [7]:
#merged.write(base_data_folder + 'retina.merged.h5ad')

In [8]:
del adata_a

In [9]:
del adata_b

In [10]:
merged

AnnData object with n_obs × n_vars = 3177310 × 8849
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster', 'AC_celltype_number', 'BC_subclass', 

In [11]:
    result_folder = "split_half." + str(round(time.time()))
    result_folder = os.path.join(base_data_folder, result_folder)
    os.mkdir(result_folder)


In [12]:
result_folder

'/vault/lfrench/temp_transfer/retina_cellxgene/split_half.1775145180'

In [13]:
    start_time = time.time()

In [ ]:
    print(result_folder)
    pymn.MetaNeighborUS(merged,
                study_col='study_id',
                ct_col='cell.type',
                fast_version=True, symmetric_output=True)
    print("After running metaneighbor all vs all")
    aurocs = merged.uns["MetaNeighborUS"]
    aurocs.to_csv(result_folder + "/aurocs_full.csv.gz", compression="gzip")

    #run 1 vs best
    pymn.MetaNeighborUS(merged,
                        study_col='study_id',
                        ct_col='cell.type', one_vs_best=True,
                        fast_version=True, symmetric_output=True)
    aurocs = merged.uns["MetaNeighborUS_1v1"]
    aurocs.to_csv(result_folder + "/aurocs_1v1.csv.gz", compression="gzip")
    print("After running metaneighbor 1 vs 1")

    cell_counts = merged.obs.groupby("study_id").size()
    cell_counts.to_csv(result_folder + "/cell_study_counts.csv")
    cell_type_counts = merged.obs[["study_id", "cell.type"]].drop_duplicates().groupby("study_id").size()
    cell_type_counts.to_csv(result_folder + "/cell_type_per_study_counts.csv")

    for set_threshold in [0.95, 0.99, 0.999]:
        print(set_threshold)
        pymn.topHits(merged, threshold=set_threshold)
        tophit_table = merged.uns['MetaNeighborUS_topHits']
        tophit_table.to_csv(result_folder + "/top_hits."+str(set_threshold)+".csv")

    print("After top hits")
    merged.obs.to_csv(result_folder + "/merged.obs.csv.zip", compression="gzip")
    merged.var.to_csv(result_folder + "/merged.var.csv.zip", compression="gzip")

    #write out peak memory at end
    mem_usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # print the memory usage in megabytes
    print("Peak memory use in Gb  " + str(round(mem_usage / 1024 / 1024,2 )) + " PID  " + str(os.getpid()))

    os.mkdir(os.path.join(result_folder, "Peak memory use in Gb " + str(round(mem_usage / 1024 / 1024 ,2))))

    end_time = time.time()
    print("Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0])
    os.mkdir(os.path.join(result_folder, "Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0]))


/vault/lfrench/temp_transfer/retina_cellxgene/split_half.1775145180
After running metaneighbor all vs all


In [15]:
end_time

1775152831.4120076

In [16]:
    print("Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0])


Time taken h_m_s 2_07_31
